# ЮрТэг — Файнтюнинг Qwen 2.5 7B

Обучение модели извлечения метаданных из юридических документов.

**Этап 1:** SFT (Supervised Fine-Tuning) на 874 одобренных примерах  
**Этап 2:** DPO (Direct Preference Optimization) на 79 парах правильный/неправильный

---

## Подготовка

1. Загрузи папку `training/` в Google Drive (или через боковую панель Colab)
2. Убедись что Runtime → Change runtime type → **T4 GPU**
3. Запускай ячейки по порядку

In [ ]:
# Установка Unsloth (занимает ~2 минуты)
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps unsloth

In [ ]:
# Подключаем Google Drive (для загрузки датасета и сохранения модели)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Загрузка данных
# Вариант 1: Из Google Drive (скопируй папку training/ в корень Drive)
# Вариант 2: Через боковую панель Colab (Files → Upload)

import os

# Попробовать Drive, потом локально
DRIVE_PATH = "/content/drive/MyDrive/yurteg_training"
LOCAL_PATH = "/content/training"

if os.path.exists(DRIVE_PATH + "/sft_train.jsonl"):
    DATA_DIR = DRIVE_PATH
    print(f"Данные найдены в Google Drive: {DATA_DIR}")
elif os.path.exists(LOCAL_PATH + "/sft_train.jsonl"):
    DATA_DIR = LOCAL_PATH
    print(f"Данные найдены локально: {DATA_DIR}")
else:
    print("ОШИБКА: Загрузи папку training/ в Google Drive (yurteg_training/) или через боковую панель")
    print("Нужные файлы: sft_train.jsonl, sft_val.jsonl, dpo_train.jsonl, dpo_val.jsonl")

## Этап 1: SFT (Supervised Fine-Tuning)

Модель учится извлекать метаданные из юридических документов на 874 одобренных примерах.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Загружаем базовую модель Qwen 2.5 7B
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=8192,   # хватает для юридических документов
    dtype=None,            # auto-detect
    load_in_4bit=True,     # экономим память GPU
)

# Добавляем LoRA адаптеры (обучаем только ~2% параметров)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                  # ранг LoRA
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # экономим ещё память
)

print(f"Обучаемых параметров: {model.print_trainable_parameters()}")

In [ ]:
from datasets import load_dataset

# Загружаем SFT-датасет
sft_dataset = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/sft_train.jsonl",
    "validation": f"{DATA_DIR}/sft_val.jsonl",
})

print(f"Train: {len(sft_dataset['train'])} примеров")
print(f"Val:   {len(sft_dataset['validation'])} примеров")

# Посмотрим один пример
example = sft_dataset['train'][0]
print(f"\nПример (роли): {[m['role'] for m in example['messages']]}")
print(f"System: {example['messages'][0]['content'][:100]}...")
print(f"Assistant: {example['messages'][2]['content'][:200]}...")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Применяем chat template к данным
def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

sft_train_formatted = sft_dataset["train"].map(apply_chat_template)
sft_val_formatted = sft_dataset["validation"].map(apply_chat_template)

# Настройки обучения SFT
sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_train_formatted,
    eval_dataset=sft_val_formatted,
    dataset_text_field="text",
    max_seq_length=8192,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir="/content/drive/MyDrive/yurteg_checkpoints/sft",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,  # эффективный batch_size = 8
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
    ),
)

print("Запускаем SFT обучение...")

In [ ]:
# SFT обучение (~15-20 минут на T4)
sft_stats = sft_trainer.train()
print(f"\nSFT завершён!")
print(f"Train loss: {sft_stats.training_loss:.4f}")

## Этап 2: DPO (Direct Preference Optimization)

Модель учится на парах: правильный ответ (chosen) vs неправильный (rejected).  
79 пар из реальных ошибок, исправленных человеком.

In [ ]:
import os

# Проверяем наличие DPO-данных
dpo_train_path = f"{DATA_DIR}/dpo_train.jsonl"
dpo_val_path = f"{DATA_DIR}/dpo_val.jsonl"

HAS_DPO = os.path.exists(dpo_train_path)
if HAS_DPO:
    print("DPO-данные найдены, запускаем обучение")
else:
    print("DPO-данных нет, пропускаем этап 2 (только SFT)")

In [ ]:
if HAS_DPO:
    from trl import DPOTrainer, DPOConfig
    from datasets import load_dataset

    # Загружаем DPO-датасет
    dpo_dataset = load_dataset("json", data_files={
        "train": dpo_train_path,
        "validation": dpo_val_path,
    })

    print(f"DPO Train: {len(dpo_dataset['train'])} пар")
    print(f"DPO Val:   {len(dpo_dataset['validation'])} пар")

    # DPO обучение
    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,  # Unsloth оптимизирует без reference model
        tokenizer=tokenizer,
        train_dataset=dpo_dataset["train"],
        eval_dataset=dpo_dataset["validation"],
        args=DPOConfig(
            output_dir="/content/drive/MyDrive/yurteg_checkpoints/dpo",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            warmup_steps=5,
            num_train_epochs=2,
            learning_rate=5e-5,      # ниже чем SFT — тонкая подстройка
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=5,
            optim="adamw_8bit",
            seed=42,
            beta=0.1,                # сила DPO-штрафа
            report_to="none",
        ),
    )

    dpo_stats = dpo_trainer.train()
    print(f"\nDPO завершён!")
    print(f"Train loss: {dpo_stats.training_loss:.4f}")
else:
    print("DPO пропущен")

## Быстрый тест

Проверяем модель на коротком тексте перед экспортом.

In [ ]:
# Тест на примере
FastLanguageModel.for_inference(model)

test_text = """ДОГОВОР ПОДРЯДА № 15
г. Москва, 10 января 2024 г.

ООО «СтройМонтаж» (Подрядчик) и ИП Фокина Дарья Владимировна (Заказчик)
заключили настоящий договор о нижеследующем:

1. Подрядчик обязуется выполнить ремонтные работы в помещении по адресу:
г. Москва, ул. Ленина, д. 10, офис 5.
2. Стоимость работ: 450 000 руб.
3. Срок выполнения: с 15.01.2024 по 15.03.2024.
4. Неустойка за просрочку: 0.1% от стоимости за каждый день."""

messages = [
    {"role": "system", "content": "Ты — опытный юрист-аналитик. Извлеки структурированные метаданные из юридического документа. Отвечай СТРОГО чистым JSON."},
    {"role": "user", "content": f"Извлеки метаданные из текста юридического документа.\n\nТекст документа:\n{test_text}"},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.1)
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("Ответ модели:")
print(response)

## Экспорт модели

Сохраняем в формате GGUF для Ollama (будет работать на маке с 8 ГБ RAM).

In [ ]:
# Сохраняем в GGUF для Ollama (q4_k_m — хороший баланс качества и размера)
SAVE_PATH = "/content/drive/MyDrive/yurteg_model"

model.save_pretrained_gguf(
    SAVE_PATH,
    tokenizer,
    quantization_method="q4_k_m",  # ~4.5 ГБ, влезает в 8 ГБ RAM
)

print(f"\nМодель сохранена в {SAVE_PATH}")
print("Файл: unsloth.Q4_K_M.gguf")

In [ ]:
# Также сохраняем LoRA-адаптеры отдельно (на случай если захотим дообучить)
LORA_PATH = "/content/drive/MyDrive/yurteg_lora"
model.save_pretrained_merged(LORA_PATH, tokenizer, save_method="lora")
print(f"LoRA-адаптеры сохранены в {LORA_PATH}")

## Готово!

Файл `unsloth.Q4_K_M.gguf` сохранён в Google Drive.

### Как запустить на маке:

```bash
# 1. Установить Ollama (если ещё нет)
brew install ollama

# 2. Скачать .gguf файл из Google Drive на мак

# 3. Создать Modelfile
cat > Modelfile << 'EOF'
FROM ./unsloth.Q4_K_M.gguf

PARAMETER temperature 0
PARAMETER num_ctx 8192

SYSTEM """Ты — опытный юрист-аналитик. Извлеки структурированные метаданные из юридического документа.
Отвечай СТРОГО чистым JSON."""
EOF

# 4. Создать модель в Ollama
ollama create yurteg -f Modelfile

# 5. Проверить
ollama run yurteg "Извлеки метаданные: Договор аренды между ООО Альфа и ИП Иванов..."
```

После этого в config.py ЮрТэга поменять провайдер на Ollama (localhost:11434).